# Lezione 2 · Pratica: Visione artificiale I — reti convoluzionali

## Cosa farai in questa pratica

In questo notebook applichi quanto visto nella Lezione 2 sulle reti neurali convoluzionali (CNN). Completi il codice in due esercizi:

- **Esercizio 1.** Costruisci l'architettura di una CNN per classificare capi di abbigliamento nel dataset FashionMNIST e la addestri dall'inizio alla fine.
- **Esercizio 2.** Riutilizzi il modello addestrato per ispezionare visivamente i suoi errori: individui quali immagini classifica male e le mostri in una griglia.

Ogni esercizio include lo scheletro del codice con parti contrassegnate come `# TODO`, che devi completare tu. Sotto ogni esercizio trovi una soluzione collassata: prova a risolvere l'esercizio da solo prima di aprirla.

## Prerequisito

Devi aver seguito la Lezione 2 ("Visione artificiale I: reti convoluzionali"), in cui si spiega cos'è un filtro convoluzionale, come funziona il *pooling* e come i due elementi si combinano in un'architettura CNN completa.

## Come eseguire questo notebook

Aprilo in Google Colab ed esegui le celle in ordine, dall'alto verso il basso. Non installare nulla di aggiuntivo: `torch` e `torchvision` sono già preinstallati in Colab. Se hai una GPU (processore grafico) assegnata, l'addestramento è più veloce, ma funziona anche su CPU.

## Esercizio 1 · Classificatore di capi di abbigliamento per un negozio online

### Il caso

Lavori nel team dati di un negozio online di abbigliamento. Quando un venditore carica nel catalogo la foto di un capo, vuoi assegnargli automaticamente una categoria (maglietta, pantaloni, cappotto, borsa...) per velocizzare la pubblicazione e mantenere il catalogo ordinato. Come primo approccio, addestri un classificatore con il dataset FashionMNIST, che contiene immagini in scala di grigi di 28 x 28 pixel suddivise in dieci categorie di capi, e che è un sostituto ragionevole delle foto reali del tuo catalogo.

### Cosa devi fare

Completa l'architettura della rete nella classe `CNNRopa`: il numero e la dimensione degli strati convoluzionali, lo strato di *pooling* e lo strato lineare finale. Il ciclo di addestramento e la funzione di valutazione sono già scritti: non modificarli, devi solo far sì che il modello sia definito e calcoli correttamente il passo in avanti (`forward`).

### Suggerimenti

- L'immagine di ingresso ha forma `(lotto, 1, 28, 28)`: un solo canale perché è in scala di grigi.
- Usa due blocchi convoluzionali. Il primo riceve 1 canale in ingresso e produce 32 canali in uscita; il secondo riceve quei 32 canali e ne produce 64. In entrambi, usa un filtro (*kernel*) di 3 x 3 con `padding=1`, così la convoluzione è "uguale" (*same*) e non riduce la dimensione spaziale in quel passo.
- Applica uno strato di *max pooling* 2 x 2 dopo ogni blocco convoluzionale. Ogni *pooling* dimezza altezza e larghezza: 28 → 14 → 7.
- Dopo i due blocchi e i due *pooling*, la mappa di attivazioni ha forma `(64, 7, 7)`. Appiattendola ottieni un vettore di `64 * 7 * 7` valori: è la dimensione di ingresso del primo strato lineare.
- Lo strato lineare finale deve produrre tanti output (logit) quante sono le classi: dieci, una per ogni tipo di capo. Non applicarvi esplicitamente `softmax`: `nn.CrossEntropyLoss` lo include già internamente.
- Non dimenticare di applicare la non linearità ReLU (unità lineare rettificata) dopo ogni convoluzione e dopo il primo strato lineare, come hai visto nella Lezione 2.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Usa GPU si Colab te la ha asignado, si no, usa CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de entrenamiento: {device}")

# Normalizamos con la media y desviación típica habituales de FashionMNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

train_dataset = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

clases = [
    "camiseta", "pantalón", "jersey", "vestido", "abrigo",
    "sandalia", "camisa", "zapatilla", "bolso", "botín",
]

In [ ]:
class CNNRopa(nn.Module):
    def __init__(self, num_clases=10):
        super().__init__()

        # TODO: define la primera capa convolucional.
        # Recibe 1 canal de entrada (escala de grises) y produce 32 canales
        # de salida. Usa kernel de 3x3 con padding=1 (convolución "igual").
        self.conv1 = ...

        # TODO: define la segunda capa convolucional.
        # Recibe los 32 canales de conv1 y produce 64 canales de salida.
        # Mismo kernel 3x3 y padding=1.
        self.conv2 = ...

        # TODO: define la capa de max pooling.
        # Usa una ventana de 2x2 que reduzca el alto y el ancho a la mitad.
        self.pool = ...

        # Tras aplicar pool dos veces sobre una entrada de 28x28: 28 -> 14 -> 7
        # El último mapa de activaciones tiene forma (64 canales, 7, 7)

        # TODO: define la primera capa totalmente conectada.
        # Recibe el vector aplanado (64 * 7 * 7 valores) y produce 128 salidas.
        self.fc1 = ...

        # TODO: define la capa final.
        # Recibe las 128 salidas de fc1 y produce num_clases salidas (los logits).
        self.fc2 = ...

        self.dropout = nn.Dropout(0.25)  # mitiga el sobreajuste

    def forward(self, x):
        # x llega con forma (lote, 1, 28, 28)

        # TODO: aplica conv1, después ReLU, después el pooling
        x = ...  # -> (lote, 32, 14, 14)

        # TODO: aplica conv2, después ReLU, después el pooling
        x = ...  # -> (lote, 64, 7, 7)

        x = torch.flatten(x, start_dim=1)  # aplana todo menos el eje de lote

        # TODO: aplica fc1 seguido de ReLU
        x = ...

        x = self.dropout(x)

        # TODO: aplica fc2 para obtener los logits (sin activación)
        x = ...

        return x


modelo = CNNRopa(num_clases=10).to(device)
print(modelo)

<details>
<summary>💡 Vedi soluzione</summary>

```python
class CNNRopa(nn.Module):
    def __init__(self, num_clases=10):
        super().__init__()

        # Blocco convoluzionale 1: 1 canale in ingresso -> 32 filtri
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32,
                                kernel_size=3, stride=1, padding=1)

        # Blocco convoluzionale 2: 32 -> 64 filtri
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64,
                                kernel_size=3, stride=1, padding=1)

        # Max pooling 2x2: dimezza altezza e larghezza ogni volta
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Dopo due pooling 2x2 su un ingresso 28x28: 28 -> 14 -> 7
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_clases)

        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # -> (lotto, 32, 14, 14)
        x = self.pool(F.relu(self.conv2(x)))  # -> (lotto, 64, 7, 7)

        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)  # logit, senza softmax esplicito
        return x


modelo = CNNRopa(num_clases=10).to(device)
print(modelo)
```
</details>

### Addestra e valuta il modello

Questa parte è già risolta: definisce la funzione di perdita, l'ottimizzatore, il ciclo di addestramento e la valutazione, e addestra il modello per alcune epoche. Devi solo eseguire le celle. Se l'architettura del paragrafo precedente è corretta, il modello si addestra senza errori e raggiunge un'accuratezza intorno al 90 % sul set di test.

In [ ]:
# CrossEntropyLoss combina softmax + entropía cruzada en un solo paso
criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)


def entrenar_una_epoca(modelo, loader, criterio, optimizador):
    modelo.train()
    perdida_total = 0.0
    for imagenes, etiquetas in loader:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)

        optimizador.zero_grad()
        salidas = modelo(imagenes)
        perdida = criterio(salidas, etiquetas)
        perdida.backward()
        optimizador.step()

        perdida_total += perdida.item() * imagenes.size(0)
    return perdida_total / len(loader.dataset)


def evaluar(modelo, loader, criterio):
    modelo.eval()
    perdida_total = 0.0
    aciertos = 0
    with torch.no_grad():
        for imagenes, etiquetas in loader:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            salidas = modelo(imagenes)
            perdida_total += criterio(salidas, etiquetas).item() * imagenes.size(0)
            predicciones = salidas.argmax(dim=1)
            aciertos += (predicciones == etiquetas).sum().item()
    perdida_media = perdida_total / len(loader.dataset)
    exactitud = aciertos / len(loader.dataset)
    return perdida_media, exactitud


num_epocas = 5
for epoca in range(1, num_epocas + 1):
    perdida_train = entrenar_una_epoca(modelo, train_loader, criterio, optimizador)
    perdida_test, exactitud_test = evaluar(modelo, test_loader, criterio)
    print(f"Época {epoca}/{num_epocas} | "
          f"pérdida train: {perdida_train:.4f} | "
          f"pérdida test: {perdida_test:.4f} | "
          f"exactitud test: {exactitud_test:.2%}")

print(f"\nExactitud final sobre el conjunto de test: {exactitud_test:.2%}")

## Esercizio 2 · Ispeziona gli errori del classificatore

### Il caso

Il tuo modello raggiunge una buona accuratezza sul set di test, ma un 90 % di risposte corrette significa anche che sbaglia un capo su dieci. Prima di portare questo classificatore in produzione nel negozio online, il team vuole vedere esempi concreti di questi errori: confonde magliette con camicie? Stivaletti con scarpe da ginnastica? Osservare gli errori con i propri occhi aiuta a decidere se l'errore è ragionevole (capi visivamente simili) o se segnala un problema più profondo del modello.

### Cosa devi fare

Completa la funzione `obtener_ejemplos_mal_clasificados`, che scorre il set di test con il modello già addestrato nell'Esercizio 1 e salva gli esempi in cui la predizione non coincide con l'etichetta reale. La funzione `mostrar_cuadricula_errores`, che disegna questi esempi in una griglia con matplotlib, è già risolta: non modificarla.

### Suggerimenti

- `salidas.argmax(dim=1)` ti dà la classe prevista per ogni immagine del lotto.
- Confronta `predicciones` con `etiquetas` elemento per elemento per ottenere una maschera booleana delle posizioni del lotto classificate male.
- Scorri le posizioni del lotto in cui la maschera vale vero e aggiungi alla lista `errores` una tupla `(immagine, etichetta_reale, etichetta_predetta)` per ciascuna. Ricordati di spostare l'immagine su CPU (`.cpu()`) prima di salvarla, perché matplotlib non lavora con tensori su GPU.
- Interrompi la raccolta non appena la lista `errores` raggiunge `max_ejemplos`, per non accumulare più esempi di quelli che mostrerai.

In [ ]:
import matplotlib.pyplot as plt


def obtener_ejemplos_mal_clasificados(modelo, dataset, device, max_ejemplos=12):
    """Recorre el dataset y devuelve una lista de tuplas
    (imagen, etiqueta_real, etiqueta_predicha) para los ejemplos
    donde el modelo se equivoca."""
    modelo.eval()
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    errores = []

    with torch.no_grad():
        for imagenes, etiquetas in loader:
            imagenes_dev = imagenes.to(device)
            etiquetas_dev = etiquetas.to(device)
            salidas = modelo(imagenes_dev)
            predicciones = salidas.argmax(dim=1)

            # TODO: calcula una máscara booleana (forma (batch,)) que sea
            # verdadera en las posiciones donde la predicción NO coincide
            # con la etiqueta real
            mascara_error = ...

            # TODO: recorre las posiciones del lote y, para cada una donde
            # mascara_error sea verdadera, añade a `errores` la tupla
            # (imagen, etiqueta_real, etiqueta_predicha). Usa
            # imagenes[i].cpu() para la imagen y .item() para convertir
            # las etiquetas a enteros de Python. Para en cuanto
            # len(errores) alcance max_ejemplos.
            ...

            if len(errores) >= max_ejemplos:
                break

    return errores[:max_ejemplos]


def mostrar_cuadricula_errores(errores, clases, columnas=4):
    """Dibuja una cuadrícula con las imágenes mal clasificadas,
    indicando la etiqueta real y la predicha en el título de cada una."""
    filas = (len(errores) + columnas - 1) // columnas
    fig, ejes = plt.subplots(filas, columnas, figsize=(columnas * 2.2, filas * 2.4))
    ejes = ejes.flatten()

    for ax, (imagen, real, predicha) in zip(ejes, errores):
        ax.imshow(imagen.squeeze(), cmap="gray")
        ax.set_title(f"real: {clases[real]}\npred: {clases[predicha]}", fontsize=9)
        ax.axis("off")

    for ax in ejes[len(errores):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


errores = obtener_ejemplos_mal_clasificados(modelo, test_dataset, device, max_ejemplos=12)
print(f"Ejemplos mal clasificados encontrados: {len(errores)}")
mostrar_cuadricula_errores(errores, clases)

<details>
<summary>💡 Vedi soluzione</summary>

```python
def obtener_ejemplos_mal_clasificados(modelo, dataset, device, max_ejemplos=12):
    """Scorre il dataset e restituisce una lista di tuple
    (immagine, etichetta_reale, etichetta_predetta) per gli esempi
    in cui il modello sbaglia."""
    modelo.eval()
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    errores = []

    with torch.no_grad():
        for imagenes, etiquetas in loader:
            imagenes_dev = imagenes.to(device)
            etiquetas_dev = etiquetas.to(device)
            salidas = modelo(imagenes_dev)
            predicciones = salidas.argmax(dim=1)

            # Maschera booleana: vera dove la predizione sbaglia
            mascara_error = predicciones != etiquetas_dev

            # Scorriamo il lotto e salviamo ogni esempio classificato male
            indices_error = mascara_error.nonzero(as_tuple=True)[0]
            for i in indices_error:
                i = i.item()
                errores.append((
                    imagenes[i].cpu(),
                    etiquetas[i].item(),
                    predicciones[i].item(),
                ))
                if len(errores) >= max_ejemplos:
                    break

            if len(errores) >= max_ejemplos:
                break

    return errores[:max_ejemplos]


errores = obtener_ejemplos_mal_clasificados(modelo, test_dataset, device, max_ejemplos=12)
print(f"Ejemplos mal clasificados encontrados: {len(errores)}")
mostrar_cuadricula_errores(errores, clases)
```
</details>

## Per concludere

Hai costruito, addestrato e verificato una rete neurale convoluzionale dall'inizio alla fine: ne hai definito l'architettura, verificato l'accuratezza su dati che non aveva mai visto e osservato da vicino dove sbaglia. Questo ciclo completo (progettare, addestrare, valutare, ispezionare gli errori) è esattamente quello che ripeterai con architetture più sofisticate nelle prossime lezioni.

Prima di proseguire, prova a cambiare il numero di filtri di `conv1` e `conv2` nell'Esercizio 1, oppure sostituisci `nn.MaxPool2d` con `nn.AvgPool2d`, e osserva come cambiano l'accuratezza e gli errori che vedi nell'Esercizio 2.

**Condividi i tuoi risultati e i tuoi dubbi nel canale del corso prima della prossima lezione in diretta!**